# RQ3: Language markers of nightmare context predicting PHQ-9 / GAD-7This notebook follows the analysis plan: derive sentiment, emotional tone, and coherence indices from dream reports and test their associations with PHQ-9 and GAD-7 using linear mixed-effects models (random intercept per participant).

## 0. Setup- Dataset: update `DATA_PATH` if needed (default `integrated_dream_data.csv`).- Requires: `pandas`, `numpy`, `statsmodels`, `scikit-learn`, `sentence-transformers` (optional), `networkx` (optional). Install as needed.```bashpip install pandas numpy statsmodels scikit-learn sentence-transformers networkx```

In [ ]:
import reimport mathfrom pathlib import Pathimport numpy as npimport pandas as pdfrom sklearn.feature_extraction.text import TfidfVectorizerfrom sklearn.metrics.pairwise import cosine_similarity# optional coherence embeddingtry:    from sentence_transformers import SentenceTransformer    EMBED_AVAILABLE = Trueexcept Exception:    SentenceTransformer = None    EMBED_AVAILABLE = False# optional mixed modelstry:    import statsmodels.formula.api as smf    import statsmodels.api as sm    STATSMODELS_AVAILABLE = Trueexcept Exception:    smf = sm = None    STATSMODELS_AVAILABLE = Falsepd.set_option('display.max_colwidth', 140)DATA_PATH = Path('integrated_dream_data.csv')assert DATA_PATH.exists(), f"Data file not found: {DATA_PATH}"

## 1. Load and basic cleaning- keep dream reports ≥ 20 characters- drop placeholder no-dream text- coerce outcome scores to numeric

In [ ]:
raw = pd.read_csv(DATA_PATH)print('raw shape', raw.shape)raw['dream_text'] = raw['dream_text'].astype(str).str.strip()raw = raw[raw['dream_text'].str.len() >= 20].copy()placeholders = [    'no dreams', 'no dream', "don't remember", "cant remember", "cannot remember",    "haven't remembered", 'none that i remember']mask_placeholder = raw['dream_text'].str.lower().apply(lambda t: any(p in t for p in placeholders))raw = raw[~mask_placeholder].copy()raw['phq_total'] = pd.to_numeric(raw.get('phq_total'), errors='coerce')raw['gad_total'] = pd.to_numeric(raw.get('gad_total'), errors='coerce')raw = raw.dropna(subset=['phq_total','gad_total'])print('after filters', raw.shape)raw[['dream_text','phq_total','gad_total']].head()

## 2. NLP indicesWe compute three indices per report:- **Sentiment**: rule-based polarity using AFINN-style valence list (embedded). Scaled to [-5, +5] mean per token.- **Emotional tone**: proportions of emotion categories from a small NRC-like lexicon; tone score = positive - negative proportion.- **Coherence**: mean cosine similarity of adjacent sentence embeddings (Sentence-BERT if available; TF-IDF fallback).

In [ ]:
# Tiny AFINN-style lexicon (can be swapped for VADER/BERT if available)afinn = {    'love':3,'like':2,'happy':3,'joy':3,'good':2,'great':3,'safe':2,    'sad':-2,'angry':-2,'mad':-2,'bad':-2,'terrible':-3,'horrible':-3,'fear':-2,'scared':-2,    'anxious':-2,'panic':-3,'depressed':-3,'hopeless':-3,'nightmare':-2,'kill':-3,'blood':-2,'danger':-2}# Mini NRC-like lexicon (expand as needed)nrc = {    'fear': ['fear','scared','terrified','panic','anxious','horror','nightmare','afraid'],    'anger': ['angry','mad','furious','rage','hate'],    'sadness': ['sad','cry','depressed','lonely','hopeless'],    'disgust': ['disgust','gross','nausea','repulse'],    'joy': ['happy','joy','pleased','laugh','delight'],    'surprise': ['surprise','startled','shocked'],    'trust': ['trust','safe','secure','confident'],    'anticipation': ['expect','hope','wait','excited']}# helper functionsTOKEN_RE = re.compile(r"[a-zA-Z']+")def tokenize(text):    return [t.lower() for t in TOKEN_RE.findall(text)]def sentiment_score(text):    toks = tokenize(text)    if not toks:        return 0.0    vals = [afinn.get(t, 0) for t in toks]    return float(np.mean(vals))def emotion_props(text):    toks = tokenize(text)    total = len(toks) or 1    counts = {emo:0 for emo in nrc}    for t in toks:        for emo, words in nrc.items():            if t in words:                counts[emo] += 1    props = {f'emo_{emo}_prop': counts[emo]/total for emo in nrc}    tone = (counts['joy'] + counts['trust'] + counts['anticipation'] - counts['fear'] - counts['anger'] - counts['sadness'] - counts['disgust'])/total    props['emotional_tone'] = tone    props['token_count'] = total    return props# coherenceif EMBED_AVAILABLE:    sb_model = SentenceTransformer('all-MiniLM-L6-v2')    def coherence_score(text):        sentences = re.split(r'[.!?]+', text)        sentences = [s.strip() for s in sentences if len(s.strip())>3]        if len(sentences) < 2:            return np.nan        vecs = sb_model.encode(sentences, normalize_embeddings=True)        sims = cosine_similarity(vecs[:-1], vecs[1:]).diagonal()        return float(np.mean(sims))else:    def coherence_score(text):        sentences = re.split(r'[.!?]+', text)        sentences = [s.strip() for s in sentences if len(s.strip())>3]        if len(sentences) < 2:            return np.nan        vecs = TfidfVectorizer().fit_transform(sentences)        sims = cosine_similarity(vecs[:-1], vecs[1:]).diagonal()        return float(np.mean(sims))# applyraw['sentiment'] = raw['dream_text'].apply(sentiment_score)emo_df = raw['dream_text'].apply(emotion_props).apply(pd.Series)raw = pd.concat([raw.reset_index(drop=True), emo_df], axis=1)raw['coherence'] = raw['dream_text'].apply(coherence_score)feature_cols = ['sentiment','emotional_tone','coherence'] + [c for c in raw.columns if c.startswith('emo_')]raw[feature_cols + ['phq_total','gad_total']].head()

### 2.1 Handle missing coherenceReplace missing coherence (single-sentence dreams) with feature mean.

In [ ]:
for col in ['coherence']:    raw[col] = raw[col].fillna(raw[col].mean())raw[feature_cols].describe()

## 3. Mixed-effects modelsRandom intercept for participant (`participant_id`). Add language and token_count as controls. Outcomes: PHQ and GAD.

In [ ]:
if not STATSMODELS_AVAILABLE:    raise ImportError('statsmodels not installed. Please pip install statsmodels')# keep needed columnscols_needed = ['participant_id','language','phq_total','gad_total','token_count'] + feature_colsmodel_df = raw[cols_needed].dropna()# standardize predictors for interpretabilityfor col in feature_cols + ['token_count']:    model_df[col + '_z'] = (model_df[col] - model_df[col].mean()) / model_df[col].std(ddof=0)# simple language dummy (reference = first level)if model_df['language'].nunique() > 1:    lang_dummies = pd.get_dummies(model_df['language'], prefix='lang', drop_first=True)    model_df = pd.concat([model_df, lang_dummies], axis=1)    lang_terms = '+'.join(lang_dummies.columns)else:    lang_terms = ''predictors = ['sentiment_z','emotional_tone_z','coherence_z','token_count_z'] + [c+'_z' for c in model_df.columns if c.startswith('emo_') and c.endswith('_prop')] + ([lang_terms] if lang_terms else [])predictor_str = ' + '.join([p for p in predictors if p])# PHQ modelformula_phq = f"phq_total ~ {predictor_str}"print('PHQ formula:', formula_phq)md_phq = smf.mixedlm(formula_phq, model_df, groups=model_df['participant_id'])res_phq = md_phq.fit(reml=False, method='lbfgs')print(res_phq.summary())# GAD modelformula_gad = f"gad_total ~ {predictor_str}"print('GAD formula:', formula_gad)md_gad = smf.mixedlm(formula_gad, model_df, groups=model_df['participant_id'])res_gad = md_gad.fit(reml=False, method='lbfgs')print(res_gad.summary())

## 4. Interpretation helpers- Look at fixed-effect coefficients: positive β → higher symptoms with higher predictor.- Check p-values and confidence intervals.- Consider variance explained at participant vs residual.- Apply multiple-comparison correction if running many predictors (Benjamini-Hochberg).

## 5. Next extensions- Swap AFINN/NRC stubs with VADER or RoBERTa sentiment; use full NRC or GoEmotions for emotion categories.- Use better coherence: full discourse coherence metrics or entity-grid models.- Add covariates: demographics (age, gender), sleep quality, country.- Explore cross-language interactions: `sentiment * language`.- Validate indices with dual LLM prompts and human ratings; flag low-confidence cases.